# Chapter 10 Simulations — Production Stage: Grid Search, Ground-Truth Evaluation & Judge Calibration

This notebook runs three evaluations against the production pipeline defined in `my_agent/agent.py`:

1. **Grid Search** — three models × three temperatures × ten analysis scenarios × five Monte Carlo simulations = 450 total runs. Ranked by F1 (judge-compliance metric). Writes the winning configuration to `best_config.json`.
2. **Ground-Truth Evaluation** — three expert-curated scenarios from `ground_truth.csv`. Pipeline output is compared against analyst-expected argument structures via recall, level accuracy, and F1.
3. **Judge Calibration** — ten synthetic argument maps with known-correct verdicts test whether the Logic Judge correctly identifies logic rule violations, strength inflation, and missing elements.

**Why all three?** Grid search measures how well the Argument Mapping Agent satisfies the judge. Ground-truth measures whether the argument map matches what a human analyst would produce. Judge calibration measures whether the judge itself can be trusted. A lenient judge produces F1=100% on an argument map with wrong strength levels — only the ground-truth and calibration tests catch that.

**Logic rules evaluated throughout:**
- `LR-001` Evidence Chain Completeness — every conclusion traces to ≥ 2 independent evidence nodes
- `LR-002` Source Citation Requirement — every inference cites at least one `E-NNN` node
- `LR-003` Strength Calibration — STRONG requires 3+ evidence nodes from 3+ source categories
- `LR-004` Assumption Explicitness — assumptions are `A-NNN` nodes, not embedded prose
- `LR-005` Alternative Hypothesis Consideration — every conclusion has at least one `ALT-NNN`

**Thresholds:**
- Judge accuracy ≥ 80% (8/10 test cases produce the correct verdict)
- Ground truth recall ≥ 60%
- Ground truth level accuracy ≥ 70%

## 1. Setup

Load environment variables and import the pipeline. `run_pipeline()` is imported directly from `my_agent/agent.py` — it constructs fresh agent instances per call and manages its own session state, so no manual session wiring is needed in the notebook.

In [1]:
import asyncio
import json
import os
import sys
import re
import csv
import math
import random
import statistics
import time
from pathlib import Path
from html import escape as _html_escape

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import HTML, display

from dotenv import load_dotenv

# Load API key — tries local .env first, then falls back to Colab Secrets.
load_dotenv()  # walks up to project root .env
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass  # Not in Colab or secret not configured

# Add my_agent/ to path so imports resolve from the notebook
AGENT_DIR = Path("my_agent").resolve()
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

from google.adk.sessions import InMemorySessionService

from agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP, APP_NAME, USER_ID
from argument_templates import compute_verdict_metrics, TEMPLATE_VERSION
from judge_eval import (
    TEST_CASES,
    run_judge_on_argument_map,
    _score_case,
)

print(f"Default model  : {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}")
print(f"Template ver   : {TEMPLATE_VERSION}")
print(f"Judge cases    : {len(TEST_CASES)}")

Default model  : gemini-2.5-flash @ temp=0.2
Template ver   : 1.0.0
Judge cases    : 10


/Users/austinn/Desktop/Packt Chapter Work/.venv/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## 2. Compute Metrics

Three judge-compliance metrics derived from the final iteration's structured verdict:

- **Precision** = `valid / (valid + unverified)` × 100 — fraction of elements the judge did not flag as rule-violating
- **Coverage** = `valid / (valid + missing_critical)` × 100 — fraction of elements the judge did not flag as absent
- **F1** = harmonic mean of the two — convergence target is 100%

These are *judge-compliance* metrics. The ground-truth section provides the independent external reference.

In [2]:
def _compute_metrics(iterations: list[dict]) -> dict:
    """
    Compute judge-compliance metrics from the final iteration of a pipeline run.

    Delegates to compute_verdict_metrics() in argument_templates.py — the
    single source of truth for P/C/F1 computation.

    Convergence tracking:
        converged=True means the pipeline reached PASS before exhausting
        max_iterations. converged=False covers both FAIL at cap and PASS on
        the last allowed iteration.
    """
    MAX_ITERS = 3  # Must match grid search constant

    if not iterations:
        return {
            "precision": 0, "coverage": 0, "f1": 0, "relevance": 0,
            "iters": 0, "passed": False, "converged": False,
            "judge_overrides": 0, "convergence_regressions": 0,
        }

    final = iterations[-1]["verdict"]
    base = compute_verdict_metrics(final)
    passed = final.get("verdict") == "PASS"

    judge_overrides = sum(
        1 for it in iterations
        if "Deterministic rule check failed" in it.get("verdict", {}).get("summary", "")
    )

    converged = passed and len(iterations) < MAX_ITERS

    convergence_regressions = sum(
        1 for it in iterations if it.get("convergence_regressed", False)
    )

    return {
        **base,
        "iters": len(iterations),
        "passed": passed,
        "converged": converged,
        "judge_overrides": judge_overrides,
        "convergence_regressions": convergence_regressions,
    }

## 3. Pipeline Helper

Wraps a single end-to-end run of `run_pipeline()` from `agent.py`. The agent constructs fresh instances per call — reusing instances across calls would raise "Agent already has a parent" from ADK's sub-agent registration.

The `run_scenario()` helper returns `(session_id, iterations, metrics)`. Metrics are computed by `_compute_metrics()` defined in the previous section.

In [3]:
async def run_scenario(
    verified_analysis: str,
    threat_context: str = "",
    max_iterations: int = 3,
    verbose: bool = True,
) -> tuple[str, list[dict], dict]:
    """
    Run the pipeline on one scenario and return (session_id, iterations, metrics).

    Prints per-iteration output when verbose=True.
    """
    _svc = InMemorySessionService()

    if verbose:
        print(f"Model: {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}  max_iter={max_iterations}")
        print()

    session_id, iterations = await run_pipeline(
        verified_analysis=verified_analysis,
        max_iterations=max_iterations,
        threat_context=threat_context,
        session_service=_svc,
    )

    if verbose:
        for it in iterations:
            n       = it["iteration"]
            verdict = it["verdict"]
            n_v     = len(verdict.get("confirmed_valid", []))
            n_u     = len(verdict.get("unverified", []))
            n_m     = len(verdict.get("missing_critical", []))
            label   = verdict.get("verdict", "UNKNOWN")
            print('─' * 60)
            print(f"Iteration {n} \u2014 {label}")
            print(f"  valid={n_v}  unverified={n_u}  missing={n_m}")
            print(f"  {verdict.get('summary', '')[:120]}")
            print()
            print("Argument Map (first 1 500 chars):")
            print(it.get("argument_map", "")[:1500])
            print()

        m = _compute_metrics(iterations)
        final_v = iterations[-1]["verdict"].get("verdict", "UNKNOWN")
        print(f"{'='*60}")
        print(f"Final verdict : {final_v} in {m['iters']} iteration(s)")
        print(f"Precision     : {m['precision']:.1f}%")
        print(f"Coverage      : {m['coverage']:.1f}%")
        print(f"F1            : {m['f1']:.1f}%")

    return session_id, iterations, _compute_metrics(iterations)

## 4. Single-Scenario Demo

Runs the AiTM Session Hijacking — Full Argument scenario: the canonical threat for ApexCode's partner access environment. The verified analysis below is the Stage 4 output the pipeline is designed to consume.

This demonstrates the full self-refining loop end-to-end: Argument Mapping Agent drafts the argument map, Logic Judge emits a structured JSON verdict, Verification Agent applies corrections, and the loop exits on PASS via the Verification Agent’s deterministic exit callback.

In [ ]:
# NOTE: REQUIRES API KEY - single pipeline run (up to ~9 model calls).
# The demo input is the canonical AiTM scenario exactly as stored in
# ground_truth.csv (scenario 1) - the same Stage 4 verified analysis the
# ground-truth evaluation consumes (the CSV stores it JSON-encoded).
import csv, json

with open("ground_truth.csv", newline="", encoding="utf-8") as _f:
    _demo = next(csv.DictReader(_f))

_analysis = json.loads(_demo["verified_analysis"])
print(f"Scenario: {_demo['Scenario']}\n")
print(_analysis.strip()[:400] + "\n[... full analysis passed to the pipeline ...]\n")

demo_session_id, demo_iterations, demo_metrics = await run_scenario(
    verified_analysis=_analysis,
    verbose=True,
)

## 5. Grid Search

The sweep covers three models × three temperatures × ten analysis scenarios × five Monte Carlo simulations = 450 total runs, capped at 5 concurrent pipelines via `asyncio.Semaphore`. Each scenario exercises a distinct threat vector so the winning configuration generalises beyond the AiTM canonical case.

The Logic Judge and Verification Agent are always fixed at `gemini-2.5-flash @ temp=0.0`. Only the Argument Mapping Agent configuration varies.

Ranked by average F1; ties broken by average iteration count (fewer iterations = faster convergence), then by average latency. The winning configuration is written to `best_config.json` and loaded by `agent.py` at import time.

**Concurrency:** `CONCURRENCY_LIMIT = 5` concurrent pipeline runs (15 in-flight Gemini requests). Uses `InMemorySessionService` to avoid SQLite WAL write-lock contention at high concurrency.

In [ ]:
import math
import random
import statistics
import time
from html import escape as _html_escape

from google.adk.sessions import InMemorySessionService

from agent import run_pipeline, APP_NAME, USER_ID
from argument_templates import compute_verdict_metrics, TEMPLATE_VERSION

# One shared in-memory session service for all grid-search runs.
# Grid search creates 450 sessions with unique UUIDs — none require cross-run
# persistence. Using InMemory avoids SQLite WAL write-lock contention: SQLite
# allows only one writer at a time (busy_timeout=0ms default), so 20 concurrent
# run_pipeline() calls against sessions.db would serialize on write locks and
# produce OperationalError: database is locked under sustained load.
_GRID_SESSION_SERVICE = InMemorySessionService()

# ---------------------------------------------------------------------------
# Grid Definition
# ---------------------------------------------------------------------------

MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.1-pro-preview",
]

TEMPERATURES = [0.0, 0.5, 1.0]

N_SIMULATIONS = 5   # Monte Carlo draws per (scenario x config) cell

SCENARIOS = [
    # 1. AiTM with full evidence chain
    "Stage 4 ACH analysis of AiTM session hijacking via Evilginx2. Three corroborating signals: Okta session anomaly from Tor exit node (identity_provider, HIGH), SpyCloud stolen session cookie (breach_intelligence, HIGH), Proofpoint phishing click (email_security, HIGH). GitHub clone of the two Project Phoenix repos from same IP (version_control, HIGH). H1 (AiTM) consistent with all evidence, H2 (legitimate VPN) rejected by partner confirmation. Recommended confidence: HIGH.",
    # 2. Ransomware with single indicator
    "Stage 4 Indicators of Change analysis of ransomware initial access. Single high-confidence indicator: CrowdStrike Cobalt Strike beacon detection (edr_xdr). Supporting signals: Okta VPN auth from unusual IP at 02:00 UTC (identity_provider, MEDIUM), HIBP breach hit for partner admin (breach_intelligence, MEDIUM). No network-level C2 confirmation. Recommended confidence: HIGH from 3 categories.",
    # 3. Insider threat with circumstantial evidence
    "Stage 4 Key Assumptions Check for insider threat. Three circumstantial signals: Netskope bulk download 500+ files (casb, MEDIUM), Salesforce 10,000-row report export (saas_monitoring, MEDIUM), GitHub clone of 5 repos (version_control, LOW). Key assumption: employee submitted resignation (confirmed with HR). No direct exfiltration to external destination observed. Recommended confidence: MEDIUM.",
    # 4. Supply chain with indirect evidence
    "Stage 4 Red Team analysis of supply chain compromise. Evidence across 2 partner organizations: Partner A GitHub audit log shows unauthorized PAT creation (version_control), Partner B Okta shows session from same IP as Partner A anomaly (identity_provider). CrowdStrike shows no endpoint compromise. IntelligenceX dark web mention of 'SaaS developer access' (dark_web, LOW). Attribution insufficient — no infrastructure match. Recommended confidence: MEDIUM.",
    # 5. BEC wire fraud
    "Stage 4 What-If analysis of BEC wire fraud attempt. $2M transfer intercepted. M365 UAL shows New-InboxRule for forwarding (collaboration, HIGH), Entra ID Audit shows OAuth consent to unknown app (identity_provider, HIGH), Entra ID Sign-In from new IP+device (identity_provider, MEDIUM), Proofpoint credential harvesting click (email_security, HIGH), Netskope DLP incident on forwarded emails (casb, MEDIUM). Full chain from phishing to exfiltration. Recommended confidence: HIGH.",
    # 6. Cloud S3 exposure with regulatory implications
    "Stage 4 Structured Brainstorming on cloud S3 bucket exposure. AWS CloudTrail shows PutBucketPolicy making bucket public (cloud_audit, HIGH). AWS CloudTrail shows GetObject calls from unknown IPs (cloud_audit, MEDIUM). No evidence of targeted exploitation vs. automated scanning. Data classification: PII present per DLP scan. Regulatory disclosure may be required under GDPR/CCPA. Recommended confidence: MEDIUM for targeted vs. HIGH for exposure.",
    # 7. Credential stuffing with breach correlation
    "Stage 4 ACH analysis of credential stuffing campaign. HIBP confirms partner domain in recent breach (breach_intelligence, MEDIUM). Okta System Log shows surge of failed authentications from distributed IPs (identity_provider, HIGH). GreyNoise classifies source IPs as known credential stuffing botnet (ip_reputation, MEDIUM). A subset of attempts succeeded before lockout triggered. Recommended confidence: HIGH from 3 categories.",
    # 8. Spear phishing with info-stealer delivery
    "Stage 4 Indicators of Change analysis of spear phishing campaign. Proofpoint blocked email with info-stealer attachment targeting engineering team (email_security, HIGH). Microsoft Defender for Endpoint detected info-stealer execution on one endpoint (edr_xdr, HIGH). CrowdStrike shows C2 callback from same endpoint (edr_xdr, HIGH — but same category as Defender). Palo Alto THREAT event for C2 IP (network_security, MEDIUM). Recommended confidence: HIGH from 3 categories (email_security, edr_xdr, network_security).",
    # 9. API key exposure with confirmed exploitation
    "Stage 4 Red Team analysis of API key exposure. GitHub Public Events API shows push with AWS access key in plaintext (code_exposure, HIGH). AWS CloudTrail shows GetSecretValue and DescribeInstances from unknown IP using exposed key within 90 minutes (cloud_audit, HIGH). HashiCorp Vault token renewal for associated service account (pam, MEDIUM). VirusTotal flags unknown IP as malicious (threat_intelligence, LOW). Four evidence items from four categories. Recommended confidence: HIGH.",
    # 10. Lateral movement through privilege escalation
    "Stage 4 ACH analysis of lateral movement. CyberArk PSM.Session.Start for high-value database safe by unapproved account (pam, HIGH). Okta System Log shows group membership change adding account to 'DB-Admins' 2 hours prior (identity_provider, HIGH). AWS CloudTrail shows AssumeRole to admin role from non-corporate IP (cloud_audit, MEDIUM). ServiceNow shows no change request. Three categories corroborate. Recommended confidence: HIGH.",
]

THREAT_CONTEXT = (
    "Transform the following Stage 4 analysis into a structured intelligence product. "
    "Build evidence chains, assign inference strength levels per logic rules."
)

MAX_ITERATIONS = 3

# Maximum concurrent run_pipeline() calls across all configs/scenarios/sims.
# Set to 5: each pipeline makes 3 async LLM calls (argument → judge → verify),
# so 5 concurrent pipelines = 15 in-flight Gemini requests — within typical
# quota limits without triggering rate-limit backoff cascades.
# Grid search uses _GRID_SESSION_SERVICE (InMemory) so SQLite WAL write-lock
# contention is not a concern here; this limit exists to manage API rate limits.
CONCURRENCY_LIMIT = 5

# Structured checkpoint file — replaces fragile regex-based log parsing.
# Each completed run is appended as a JSON line for reliable resume.
_CHECKPOINT_PATH = Path("config_search_checkpoint.jsonl")

# Display short-name for each model.
_MODEL_SHORT: dict[str, str] = {
    "gemini-2.5-flash":       "flash",
    "gemini-2.5-pro":         "pro",
    "gemini-3.1-pro-preview": "3.1-pro",
}

# Reverse map: log short-name -> full model name (for checkpoint parsing).
_SHORT_TO_MODEL: dict[str, str] = {v: k for k, v in _MODEL_SHORT.items()}
_SHORT_TO_MODEL["preview"] = "gemini-3.1-pro-preview"

# ---------------------------------------------------------------------------
# Metrics — single pipeline run
# ---------------------------------------------------------------------------


def _compute_metrics(iterations: list[dict]) -> dict:
    """
    Compute judge-compliance metrics from the final iteration of a pipeline run.

    Delegates to compute_verdict_metrics() in argument_templates.py — the
    single source of truth for P/C/F1 computation.

    Convergence tracking:
        converged=True means the pipeline reached PASS before exhausting
        max_iterations. converged=False covers both FAIL at cap and PASS on
        the last allowed iteration. This distinction reveals whether max_iterations
        is sufficient: if converged_rate is low, the cap may be truncating runs
        that needed one more pass.
    """
    if not iterations:
        return {
            "precision": 0, "coverage": 0, "f1": 0, "relevance": 0,
            "iters": 0, "passed": False, "converged": False,
            "judge_overrides": 0, "convergence_regressions": 0,
        }

    final = iterations[-1]["verdict"]
    base = compute_verdict_metrics(final)
    passed = final.get("verdict") == "PASS"

    # Count how many iterations had the deterministic rule checker override
    # the judge verdict (PASS downgraded to FAIL).
    judge_overrides = sum(
        1 for it in iterations
        if "Deterministic rule check failed" in it.get("verdict", {}).get("summary", "")
    )

    # converged=True only when the pipeline passed BEFORE hitting the iteration
    # cap. A run that passes on the final allowed iteration is not "converged
    # early" — we don't know if it would have passed sooner with more iterations.
    converged = passed and len(iterations) < MAX_ITERATIONS

    # Count convergence regressions: iterations where unverified count increased
    # vs. the prior iteration (tracked per-iteration in agent.py).
    convergence_regressions = sum(
        1 for it in iterations if it.get("convergence_regressed", False)
    )

    return {
        **base,
        "iters": len(iterations),
        "passed": passed,
        "converged": converged,
        "judge_overrides": judge_overrides,
        "convergence_regressions": convergence_regressions,
    }


# ---------------------------------------------------------------------------
# Aggregation — N simulations → one scenario-level result
# ---------------------------------------------------------------------------


def _aggregate_simulations(sim_results: list[dict]) -> dict:
    """Average N simulation results into one scenario-level aggregate."""
    n = len(sim_results)
    if n == 0:
        return {
            "precision": 0, "coverage": 0, "f1": 0, "f1_std": 0,
            "iters": 0, "sim_pass_rate": 0, "passed": False,
            "avg_latency_s": 0, "latency_std_s": 0,
        }

    f1s           = [r["f1"] for r in sim_results]
    precisions    = [r["precision"] for r in sim_results]
    coverages     = [r["coverage"] for r in sim_results]
    iters_list    = [r["iters"] for r in sim_results]
    passes        = [r["passed"] for r in sim_results]
    converged     = [r.get("converged", False) for r in sim_results]
    regressions   = [r.get("convergence_regressions", 0) for r in sim_results]
    latencies     = [r.get("latency_s", 0) for r in sim_results]

    return {
        "precision":               round(statistics.mean(precisions), 1),
        "coverage":                round(statistics.mean(coverages), 1),
        "f1":                      round(statistics.mean(f1s), 1),
        "f1_std":                  round(statistics.stdev(f1s), 2) if n > 1 else 0.0,
        "iters":                   round(statistics.mean(iters_list), 1),
        "sim_pass_rate":           round(sum(passes) / n, 2),
        "convergence_rate":        round(sum(converged) / n, 2),
        "avg_convergence_regressions": round(statistics.mean(regressions), 2),
        "passed":                  sum(passes) / n > 0.5,
        "avg_latency_s":           round(statistics.mean(latencies), 1),
        "latency_std_s":           round(statistics.stdev(latencies), 1) if n > 1 else 0.0,
    }


# ---------------------------------------------------------------------------
# Checkpoint / resume
# ---------------------------------------------------------------------------

_counter_lock = asyncio.Lock()
_checkpoint_lock = asyncio.Lock()


def _load_structured_checkpoint() -> dict:
    """Load checkpoint from structured JSONL file (one JSON object per line).

    Returns {(model, temp, scenario_idx, sim_idx): metrics_dict}.
    More reliable than regex-based log parsing — survives format changes.
    """
    results: dict = {}
    if not _CHECKPOINT_PATH.exists():
        return results
    n_loaded = 0
    for line in _CHECKPOINT_PATH.read_text(errors="replace").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
            key = (r["model"], r["temperature"], r["scenario_idx"], r["sim_idx"])
            # Skip API-error FAILs (F1=0, latency=0) so they retry.
            if r.get("f1", 0) == 0 and r.get("latency_s", 0) == 0:
                continue
            results[key] = r
            n_loaded += 1
        except (json.JSONDecodeError, KeyError):
            continue
    if n_loaded:
        print(f"  [checkpoint] Loaded {n_loaded} runs from {_CHECKPOINT_PATH.name}")
    return results


async def _save_checkpoint_entry(result: dict) -> None:
    """Append a single result to the structured JSONL checkpoint file."""
    async with _checkpoint_lock:
        with open(_CHECKPOINT_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")


async def _run_single_sim(
    sem: asyncio.Semaphore,
    scenario: str,
    sc_idx: int,
    sim_idx: int,
    temp: float,
    model: str,
    counter: dict,
    max_retries: int = 3,
) -> dict:
    """Run one pipeline simulation with concurrency limiting and retry."""
    async with sem:
        start = time.monotonic()
        for attempt in range(1, max_retries + 1):
            try:
                session_id, iterations = await run_pipeline(
                    verified_analysis=scenario,
                    max_iterations=MAX_ITERATIONS,
                    production_temp=temp,
                    production_model=model,
                    threat_context=THREAT_CONTEXT,
                    session_service=_GRID_SESSION_SERVICE,
                )
                elapsed = time.monotonic() - start
                metrics = _compute_metrics(iterations)
                metrics["latency_s"] = round(elapsed, 1)

                # Delete the session immediately after metrics
                # are extracted. Each run_pipeline() call creates a session with
                # a unique UUID. Without deletion, all 450 sessions accumulate
                # in _GRID_SESSION_SERVICE for the lifetime of the process,
                # holding argument_map (~3 KB), verified_product (~5 KB), and
                # full event history per session. Deleting promptly bounds memory
                # to CONCURRENCY_LIMIT live sessions at any time (~5 × ~30 KB).
                try:
                    await _GRID_SESSION_SERVICE.delete_session(
                        app_name=APP_NAME,
                        user_id=USER_ID,
                        session_id=session_id,
                    )
                except Exception:
                    pass  # Non-fatal — metrics are already recorded.

                async with _counter_lock:
                    counter["done"] += 1
                    n = counter["done"]
                    total = counter["total"]

                short = _MODEL_SHORT.get(model, model)
                verdict_tag = "PASS" if metrics["passed"] else "FAIL"
                print(
                    f"  [{n:>3}/{total}]  {short:<8s} t={temp}  "
                    f"SC-{sc_idx+1:<2d} sim{sim_idx+1}  →  "
                    f"F1={metrics['f1']:.0f}%  P={metrics['precision']:.0f}%  "
                    f"C={metrics['coverage']:.0f}%  "
                    f"iters={metrics['iters']}  {elapsed:.0f}s  {verdict_tag}"
                )
                result = {
                    "model": model, "temperature": temp,
                    "scenario_idx": sc_idx, "sim_idx": sim_idx,
                    **metrics,
                }
                await _save_checkpoint_entry(result)
                return result
            except (TypeError, KeyError, AttributeError) as exc:
                # Deterministic failure — don't retry, record as zero.
                async with _counter_lock:
                    counter["done"] += 1
                print(
                    f"  [FAIL-NORETRY] {model} t={temp} SC-{sc_idx+1} "
                    f"sim{sim_idx+1}: {type(exc).__name__}: {exc}"
                )
                return {
                    "model": model, "temperature": temp,
                    "scenario_idx": sc_idx, "sim_idx": sim_idx,
                    "precision": 0, "coverage": 0, "f1": 0,
                    "relevance": 0, "iters": 0, "passed": False, "latency_s": 0,
                    "judge_overrides": 0, "convergence_regressions": 0,
                }
            except Exception as exc:
                # Transient failure (API, network, rate limit) — retry with jitter.
                if attempt == max_retries:
                    async with _counter_lock:
                        counter["done"] += 1
                    print(f"  [FAIL] {model} t={temp} SC-{sc_idx+1} sim{sim_idx+1}: {exc}")
                    return {
                        "model": model, "temperature": temp,
                        "scenario_idx": sc_idx, "sim_idx": sim_idx,
                        "precision": 0, "coverage": 0, "f1": 0,
                        "relevance": 0, "iters": 0, "passed": False, "latency_s": 0,
                        "judge_overrides": 0,
                    }
                wait = 2 ** attempt * 5 * (0.5 + random.random())
                await asyncio.sleep(wait)


# ---------------------------------------------------------------------------
# Config result builder
# ---------------------------------------------------------------------------


def _build_config_result(model: str, temp: float, flat_results: list[dict]) -> dict:
    """Build per-config aggregate from flat simulation results."""
    # Group by scenario.
    by_scenario: dict[int, list[dict]] = {}
    for r in flat_results:
        sc = r["scenario_idx"]
        by_scenario.setdefault(sc, []).append(r)

    scenario_results = []
    for sc_idx in sorted(by_scenario.keys()):
        agg = _aggregate_simulations(by_scenario[sc_idx])
        agg["scenario_idx"] = sc_idx
        scenario_results.append(agg)

    # Config-level averages — ALL runs included, including failures.
    # Previous implementation excluded F1=0 runs, creating survivorship bias:
    # configs that crashed frequently appeared to perform better because only
    # their successful runs contributed to the average. Now failed runs
    # contribute F1=0, so unreliable configs are properly penalized.
    all_f1s = [r["f1"] for r in flat_results]
    all_precisions = [r["precision"] for r in flat_results]
    all_coverages = [r["coverage"] for r in flat_results]
    all_iters = [r["iters"] for r in flat_results if r["iters"] > 0]
    all_latencies = [r["latency_s"] for r in flat_results if r["latency_s"] > 0]
    n_pass = sum(1 for r in flat_results if r["passed"])
    n_converged = sum(1 for r in flat_results if r.get("converged", False))
    n_total = len(flat_results)

    avg_f1 = round(statistics.mean(all_f1s), 1) if all_f1s else 0.0
    se_f1 = round(statistics.stdev(all_f1s) / (len(all_f1s) ** 0.5), 2) if len(all_f1s) > 1 else 0.0

    # Judge override rate: how often the deterministic rule checker
    # caught errors the LLM judge missed. High override rate = unreliable judge.
    all_overrides = [r.get("judge_overrides", 0) for r in flat_results]
    avg_overrides = round(statistics.mean(all_overrides), 2) if all_overrides else 0.0

    # Convergence regression rate: iterations where unverified count increased
    # vs. prior iteration. Non-zero = loop is not monotonically contracting.
    all_regressions = [r.get("convergence_regressions", 0) for r in flat_results]
    avg_regressions = round(statistics.mean(all_regressions), 2) if all_regressions else 0.0

    return {
        "model":                model,
        "temperature":          temp,
        "scenario_results":     scenario_results,
        "avg_precision":        round(statistics.mean(all_precisions), 1) if all_precisions else 0.0,
        "avg_coverage":         round(statistics.mean(all_coverages), 1) if all_coverages else 0.0,
        "avg_f1":               avg_f1,
        "avg_f1_std":           round(statistics.stdev(all_f1s), 2) if len(all_f1s) > 1 else 0.0,
        "se_avg_f1":            se_f1,
        "avg_iters":            round(statistics.mean(all_iters), 1) if all_iters else 0.0,
        "pass_rate":            round(n_pass / n_total, 2) if n_total > 0 else 0.0,
        # convergence_rate: fraction of runs that reached PASS before the iteration
        # cap. A pass on the final iteration is not early convergence. Low convergence
        # rate with high pass_rate means most runs squeeze a PASS out on the last
        # iteration — the cap may be too tight.
        "convergence_rate":     round(n_converged / n_total, 2) if n_total > 0 else 0.0,
        "avg_latency_s":        round(statistics.mean(all_latencies), 1) if all_latencies else 0.0,
        "latency_std_s":        round(statistics.stdev(all_latencies), 1) if len(all_latencies) > 1 else 0.0,
        "avg_judge_overrides":          avg_overrides,
        "avg_convergence_regressions":  avg_regressions,
    }


# ---------------------------------------------------------------------------
# Grid runner
# ---------------------------------------------------------------------------


async def run_grid(checkpoint: dict | None = None) -> list[dict]:
    """Run the full grid search, skipping checkpointed runs."""
    checkpoint = checkpoint or {}

    # Build all keys.
    all_keys = [
        (model, temp, sc_idx, sim_idx)
        for model in MODELS
        for temp in TEMPERATURES
        for sc_idx in range(len(SCENARIOS))
        for sim_idx in range(N_SIMULATIONS)
    ]

    # Filter to missing keys.
    missing_keys = [k for k in all_keys if k not in checkpoint]
    total_runs = len(all_keys)
    cached_runs = total_runs - len(missing_keys)

    print(f"\n  Total grid cells:  {total_runs}")
    print(f"  Cached (skipped):  {cached_runs}")
    print(f"  Runs to execute:   {len(missing_keys)}")

    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    counter = {"done": cached_runs, "total": total_runs}

    # Launch all missing runs.
    tasks = [
        _run_single_sim(sem, SCENARIOS[sc_idx], sc_idx, sim_idx, temp, model, counter)
        for model, temp, sc_idx, sim_idx in missing_keys
    ]

    new_results = await asyncio.gather(*tasks)

    # Merge checkpoint + new results.
    all_results: dict[tuple, dict] = dict(checkpoint)
    for r in new_results:
        key = (r["model"], r["temperature"], r["scenario_idx"], r["sim_idx"])
        all_results[key] = r

    # Build config-level results.
    configs = []
    for model in MODELS:
        for temp in TEMPERATURES:
            flat = [
                all_results[k] for k in all_keys
                if k[0] == model and k[1] == temp and k in all_results
            ]
            if flat:
                configs.append(_build_config_result(model, temp, flat))

    return configs


# ---------------------------------------------------------------------------
# Output helpers
# ---------------------------------------------------------------------------


def _welch_t_test(
    mean1: float, std1: float, n1: int,
    mean2: float, std2: float, n2: int,
) -> tuple[float, float, bool]:
    """
    Welch's two-sample t-test (unequal variances).

    Ranking 9 configs by avg_f1 with no significance test
    treats sampling noise as a real difference. This function tests whether
    the top config is statistically distinguishable from the runner-up.

    Returns:
        (t_stat, p_value, significant)
        significant=True means p < 0.05 (two-tailed).

    Implementation:
        t-statistic via Welch formula.
        p-value via normal approximation (math.erfc), accurate for df > 30.
        With n1=n2=50 flat results per config, df ≈ 98 — the approximation
        error vs. the exact t-distribution is < 0.002.
    """
    var1_n = (std1 ** 2) / n1 if n1 > 1 else 0.0
    var2_n = (std2 ** 2) / n2 if n2 > 1 else 0.0
    se = math.sqrt(var1_n + var2_n)
    if se == 0:
        return 0.0, 1.0, False
    t_stat = (mean1 - mean2) / se
    # Two-tailed p-value via normal approximation.
    p_value = math.erfc(abs(t_stat) / math.sqrt(2))
    return t_stat, p_value, p_value < 0.05


def _print_ranked_table(configs: list[dict]) -> None:
    """Print a ranked table sorted by avg_f1 descending."""
    ranked = sorted(configs, key=lambda c: (-c["avg_f1"], c["avg_iters"]))
    print(f"\n{'='*112}")
    print("JUDGE COMPLIANCE RANKING (not ground-truth accuracy — see Section 2 above)")
    print(f"{'Rank':<5} {'Model':<25} {'Temp':<6} {'F1':>6} {'±SE':>6} "
          f"{'P':>6} {'C':>6} {'Pass%':>6} {'Conv%':>6} {'Iters':>6} {'Lat(s)':>7} {'Overrides':>9}")
    print("-" * 112)
    for i, c in enumerate(ranked, 1):
        overrides = c.get("avg_judge_overrides", 0)
        conv_rate = c.get("convergence_rate", 0)
        print(
            f"{i:<5} {c['model']:<25} {c['temperature']:<6.1f} "
            f"{c['avg_f1']:>5.1f}% {c['se_avg_f1']:>5.1f}  "
            f"{c['avg_precision']:>5.1f}% {c['avg_coverage']:>5.1f}% "
            f"{c['pass_rate']:>5.0%} {conv_rate:>5.0%} {c['avg_iters']:>5.1f}  "
            f"{c['avg_latency_s']:>6.1f} {overrides:>8.1f}"
        )
    print("=" * 112)

    # Multiple comparison correction.
    # With 9 configs ranked by avg_f1, the expected maximum of 9 independent
    # estimates exceeds the true mean even when all configs perform identically.
    # Report whether rank-1 is statistically distinguishable from rank-2.
    if len(ranked) >= 2:
        r1, r2 = ranked[0], ranked[1]
        n_per_config = N_SIMULATIONS * len(SCENARIOS)
        t_stat, p_value, significant = _welch_t_test(
            r1["avg_f1"], r1["avg_f1_std"], n_per_config,
            r2["avg_f1"], r2["avg_f1_std"], n_per_config,
        )
        gap = r1["avg_f1"] - r2["avg_f1"]
        if significant:
            print(
                f"\nRank-1 vs Rank-2: F1 gap={gap:.1f}pp  t={t_stat:.2f}  "
                f"p={p_value:.3f}  → STATISTICALLY DISTINGUISHABLE (p<0.05)"
            )
        else:
            print(
                f"\nRank-1 vs Rank-2: F1 gap={gap:.1f}pp  t={t_stat:.2f}  "
                f"p={p_value:.3f}  → STATISTICALLY TIED (p≥0.05) — "
                f"prefer lower temperature as tiebreaker"
            )


def _write_best_config(configs: list[dict]) -> Path:
    """Write best_config.json with the top-ranked configuration."""
    ranked = sorted(configs, key=lambda c: (-c["avg_f1"], c["avg_iters"]))
    best = ranked[0]

    out = {
        "model":            best["model"],
        "temperature":      best["temperature"],
        "avg_f1":           best["avg_f1"],
        "avg_f1_std":       best["avg_f1_std"],
        "avg_iters":        best["avg_iters"],
        "pass_rate":        best["pass_rate"],
        "convergence_rate": best.get("convergence_rate", 0.0),
        "avg_latency_s":    best["avg_latency_s"],
        "latency_std_s":    best["latency_std_s"],
        "n_simulations":    N_SIMULATIONS,
        "n_scenarios":      len(SCENARIOS),
        # Schema version at search time. If argument_templates.TEMPLATE_VERSION
        # changes after this file is written, agent.py will warn at import.
        "template_version": TEMPLATE_VERSION,
    }

    out_path = Path("my_agent/best_config.json")
    out_path.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print(f"\nWrote best config: {out_path}")
    print(f"  Model: {best['model']}  Temp: {best['temperature']}  "
          f"F1: {best['avg_f1']:.1f}%  Pass: {best['pass_rate']:.0%}")
    return out_path


# ---------------------------------------------------------------------------
# HTML Report
# ---------------------------------------------------------------------------


def _html_search_sankey(best: dict) -> str:
    """SVG Sankey diagram showing the config search pipeline flow."""
    W, H = 960, 270
    nodes = [
        (30,  80, 140, 110, "Grid Setup",       f"{len(MODELS)}×{len(TEMPERATURES)}"),
        (220, 60, 140, 150, "Monte Carlo",      f"×{N_SIMULATIONS} sims"),
        (420, 40, 140, 190, "Pipeline Exec",    f"{len(MODELS)*len(TEMPERATURES)*len(SCENARIOS)*N_SIMULATIONS} runs"),
        (620, 70, 140, 130, "Aggregation",      "avg F1, SE"),
        (810, 90, 120, 90,  "Best Config",      f"{best.get('model','?')}"),
    ]
    colors = ["#6366f1", "#0891b2", "#d97706", "#16a34a"]

    svg = [f'<svg viewBox="0 0 {W} {H}" xmlns="http://www.w3.org/2000/svg" '
           f'style="width:100%;max-width:{W}px;display:block;margin:auto">']

    # Bands between nodes.
    for i in range(len(nodes) - 1):
        x1 = nodes[i][0] + nodes[i][2]
        y1_mid = nodes[i][1] + nodes[i][3] / 2
        x2 = nodes[i+1][0]
        y2_mid = nodes[i+1][1] + nodes[i+1][3] / 2
        bw = 30
        cx1 = x1 + 40
        cx2 = x2 - 40
        c = colors[i]
        svg.append(
            f'<path d="M{x1},{y1_mid-bw/2} C{cx1},{y1_mid-bw/2} {cx2},{y2_mid-bw/2} {x2},{y2_mid-bw/2} '
            f'L{x2},{y2_mid+bw/2} C{cx2},{y2_mid+bw/2} {cx1},{y1_mid+bw/2} {x1},{y1_mid+bw/2} Z" '
            f'fill="{c}" opacity="0.22"/>'
        )

    # Node rectangles.
    for x, y, w, h, label, sub in nodes:
        stroke = "#16a34a" if label == "Best Config" else "#334155"
        svg.append(f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="10" '
                   f'fill="#1e293b" stroke="{stroke}" stroke-width="2"/>')
        svg.append(f'<text x="{x+w/2}" y="{y+h/2-8}" text-anchor="middle" '
                   f'fill="#e2e8f0" font-size="13" font-weight="600">{label}</text>')
        fill_sub = "#86efac" if label == "Best Config" else "#94a3b8"
        svg.append(f'<text x="{x+w/2}" y="{y+h/2+12}" text-anchor="middle" '
                   f'fill="{fill_sub}" font-size="11">{sub}</text>')

    svg.append("</svg>")
    return f'<div class="card"><h2>Search Pipeline</h2>{"".join(svg)}</div>'


def _html_config_perf_table(configs: list[dict]) -> str:
    """Ranked config table with color-coded cells and top-2 significance note."""
    ranked = sorted(configs, key=lambda c: (-c["avg_f1"], c["avg_iters"]))
    rows = []
    for i, c in enumerate(ranked, 1):
        f1_color = "#16a34a" if c["avg_f1"] >= 90 else "#d97706" if c["avg_f1"] >= 70 else "#dc2626"
        conv_rate = c.get("convergence_rate", 0)
        # Conv% < pass_rate means some runs only passed on the final iteration —
        # the cap may be too tight for this config.
        conv_color = "#16a34a" if conv_rate >= 0.8 else "#d97706" if conv_rate >= 0.5 else "#dc2626"
        rows.append(
            f'<tr><td>{i}</td><td>{_html_escape(c["model"])}</td><td>{c["temperature"]}</td>'
            f'<td style="color:{f1_color};font-weight:700">{c["avg_f1"]:.1f}%</td>'
            f'<td>{c["se_avg_f1"]:.1f}</td>'
            f'<td>{c["avg_precision"]:.1f}%</td><td>{c["avg_coverage"]:.1f}%</td>'
            f'<td>{c["pass_rate"]:.0%}</td>'
            f'<td style="color:{conv_color}">{conv_rate:.0%}</td>'
            f'<td>{c["avg_iters"]:.1f}</td>'
            f'<td>{c["avg_latency_s"]:.0f}s</td></tr>'
        )
    # Report top-2 significance test in the HTML table.
    sig_note = ""
    if len(ranked) >= 2:
        n_per_config = N_SIMULATIONS * len(SCENARIOS)
        t_stat, p_value, significant = _welch_t_test(
            ranked[0]["avg_f1"], ranked[0]["avg_f1_std"], n_per_config,
            ranked[1]["avg_f1"], ranked[1]["avg_f1_std"], n_per_config,
        )
        gap = ranked[0]["avg_f1"] - ranked[1]["avg_f1"]
        if significant:
            sig_note = (
                f'<p style="font-size:0.8rem;color:#16a34a;margin-top:0.8rem">'
                f"Rank-1 vs Rank-2: F1 gap={gap:.1f}pp &nbsp; t={t_stat:.2f} &nbsp; "
                f"p={p_value:.3f} &nbsp; ✓ Statistically distinguishable (p&lt;0.05)</p>"
            )
        else:
            sig_note = (
                f'<p style="font-size:0.8rem;color:#f59e0b;margin-top:0.8rem">'
                f"Rank-1 vs Rank-2: F1 gap={gap:.1f}pp &nbsp; t={t_stat:.2f} &nbsp; "
                f"p={p_value:.3f} &nbsp; ⚠ Statistically tied (p≥0.05) — "
                f"prefer lower temperature as tiebreaker</p>"
            )

    return f"""<div class="card"><h2>Configuration Performance</h2>
<p style="font-size:0.8rem;color:#94a3b8;margin-bottom:0.8rem">
  Conv% = fraction of runs that reached PASS before hitting the iteration cap ({MAX_ITERATIONS}).
  If Conv% &lt; Pass%, some runs scraped a PASS on the last iteration — the cap may be too tight.
</p>
<table><thead><tr><th>Rank</th><th>Model</th><th>Temp</th><th>F1</th><th>±SE</th>
<th>Precision</th><th>Coverage</th><th>Pass%</th><th>Conv%</th><th>Iters</th><th>Latency</th></tr></thead>
<tbody>{''.join(rows)}</tbody></table>{sig_note}</div>"""


def _html_eval_metrics() -> str:
    """Metric definition cards with self-referential measurement caveat."""
    return """<div class="card"><h2>Evaluation Metrics (Judge Compliance)</h2>
<p style="font-size:0.82rem;color:#f59e0b;margin-bottom:1rem;line-height:1.5">
  ⚠ These metrics measure <strong>judge compliance</strong>, not ground-truth
  correctness. Precision, Coverage, and F1 are computed from the judge's own
  verdict fields. The judge is both the quality gate and the metric source.
  For external validation, run <code>Section 2 above</code>.
</p>
<div class="stat-row">
  <div class="stat"><span class="label">Precision</span>
    <span class="sub">valid / (valid + unverified) — fraction of elements the judge did not flag as rule-violating</span></div>
  <div class="stat"><span class="label">Coverage</span>
    <span class="sub">valid / (valid + missing_critical) — fraction of elements the judge did not flag as absent</span></div>
  <div class="stat"><span class="label">F1</span>
    <span class="sub">Harmonic mean of Precision and Coverage — measures judge self-consistency, not correctness</span></div>
</div></div>"""


def build_search_report(configs: list[dict], best: dict) -> str:
    """Generate the full HTML search report."""
    ranked = sorted(configs, key=lambda c: (-c["avg_f1"], c["avg_iters"]))
    total_runs = len(MODELS) * len(TEMPERATURES) * len(SCENARIOS) * N_SIMULATIONS
    best_pass = best.get("pass_rate", 0)
    sankey_html = _html_search_sankey(best)
    eval_metrics_html = _html_eval_metrics()
    perf_table_html = _html_config_perf_table(ranked)
    raw_json = json.dumps(
        [{k: v for k, v in c.items() if k != "scenario_results"} for c in ranked],
        indent=2,
    )

    return f"""\
<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Config Search — Stage 5 Production</title>
<style>
  *{{margin:0;padding:0;box-sizing:border-box}}
  body{{font-family:system-ui,-apple-system,sans-serif;background:#0f172a;color:#e2e8f0;padding:1.5rem}}
  .header{{text-align:center;margin-bottom:2rem}}
  .header h1{{font-size:1.6rem;color:#f8fafc}}
  .header p{{color:#94a3b8;font-size:0.9rem}}
  .card{{background:#1e293b;border-radius:12px;padding:1.5rem;margin-bottom:1.5rem;border:1px solid #334155}}
  h2{{font-size:1.1rem;color:#f8fafc;margin-bottom:1rem}}
  .best-banner{{display:flex;flex-wrap:wrap;gap:1rem;justify-content:center;margin-bottom:2rem}}
  .stat{{background:#1e293b;border-radius:10px;padding:1rem 1.2rem;border:1px solid #334155;text-align:center;min-width:120px}}
  .stat .label{{display:block;font-size:0.75rem;color:#94a3b8;text-transform:uppercase;letter-spacing:0.05em}}
  .stat .value{{display:block;font-size:1.4rem;font-weight:700;color:#f8fafc}}
  .stat .sub{{display:block;font-size:0.7rem;color:#64748b;margin-top:0.2rem}}
  .stat-row{{display:flex;gap:1rem;flex-wrap:wrap}}
  table{{width:100%;border-collapse:collapse;font-size:0.85rem}}
  th,td{{padding:0.5rem 0.7rem;text-align:left;border-bottom:1px solid #334155}}
  th{{color:#94a3b8;font-weight:600;font-size:0.75rem;text-transform:uppercase}}
  tr:hover{{background:#334155}}
</style></head><body>

<div class="header">
  <h1>Config Search Report — Stage 5 Production</h1>
  <p>Intel Cycle AI — ApexCode Solutions</p>
  <p>Grid: {len(MODELS)} models × {len(TEMPERATURES)} temps × {len(SCENARIOS)} scenarios × {N_SIMULATIONS} sims = {total_runs} runs</p>
</div>

<div class="best-banner">
  <div class="stat"><span class="label">Best Model</span>
    <span class="value">{_html_escape(str(best.get('model','—')))}</span></div>
  <div class="stat"><span class="label">Temperature</span>
    <span class="value">{best.get('temperature','—')}</span></div>
  <div class="stat"><span class="label">avg F1</span>
    <span class="value">{best.get('avg_f1',0):.1f}%</span></div>
  <div class="stat"><span class="label">Pass Rate</span>
    <span class="value">{best_pass:.0%}</span></div>
  <div class="stat"><span class="label">Conv Rate</span>
    <span class="value">{best.get('convergence_rate',0):.0%}</span>
    <span class="sub">early-PASS / total</span></div>
  <div class="stat"><span class="label">avg Iters</span>
    <span class="value">{best.get('avg_iters',0)}</span></div>
  <div class="stat"><span class="label">avg Latency</span>
    <span class="value">{best.get('avg_latency_s',0):.0f}s</span></div>
</div>

{sankey_html}

<div class="card">
  <h2>Methodology</h2>
  <p style="font-size:0.88rem;color:#475569;line-height:1.7">
    <strong>Monte Carlo sampling:</strong> Each (scenario, config) cell runs {N_SIMULATIONS}
    independent pipeline executions. Metrics are averaged across simulations before
    aggregating across scenarios.<br><br>
    <strong>Pipeline:</strong> Argument Mapping → Logic Judge → Verification, up to
    {MAX_ITERATIONS} iterations per run. Judge and Verification held constant at
    gemini-2.5-flash @ temp=0.0 so only the Argument Mapping Agent config varies.<br><br>
    <strong>Ranking:</strong> Primary avg_F1 (judge compliance), tiebreak lower avg_iters
    (cheaper convergence). Note: F1 measures judge self-consistency, not
    ground-truth correctness. Run <code>Section 2 above</code> for external validation.
  </p>
</div>

{eval_metrics_html}
{perf_table_html}

<div class="card">
  <h2>Raw Results (JSON)</h2>
  <details>
    <summary>Expand raw config results</summary>
    <pre style="font-size:0.78rem;background:#f8fafc;color:#1e293b;padding:1rem;
                border-radius:6px;overflow-x:auto;max-height:500px">{raw_json}</pre>
  </details>
</div>

</body></html>"""


total_runs = len(MODELS) * len(TEMPERATURES) * len(SCENARIOS) * N_SIMULATIONS
print(f"Models          : {MODELS}")
print(f"Temperatures    : {TEMPERATURES}")
print(f"Scenarios       : {len(SCENARIOS)}")
print(f"Simulations     : {N_SIMULATIONS} per cell")
print(f"Total runs      : {total_runs}")
print(f"Max iterations  : {MAX_ITERATIONS} per run")
print(f"Concurrency     : {CONCURRENCY_LIMIT} parallel pipelines")
print()
print("Scenarios:")
for i, s in enumerate(SCENARIOS, 1):
    print(f"  {i:>2}. {s[:90]}...")

In [ ]:
# NOTE: REQUIRES API KEY - grid search: 3 models x 3 temps x 10 scenarios x 5 sims
# = 450 pipeline runs (roughly 6 model calls each). Run this cell yourself to
# find the best Argument Mapping Agent config for your environment. Progress is
# checkpointed to config_search_checkpoint.jsonl after every run, so an
# interrupted sweep resumes where it left off when you re-run this cell.

grid_configs = await run_grid(checkpoint=_load_structured_checkpoint())

In [ ]:
# Ranked results table, persist the winning config, and render the search report.
if not globals().get("grid_configs"):
    print("No grid results yet - run the grid search cell above first.")
else:
    _print_ranked_table(grid_configs)
    _write_best_config(grid_configs)

    _best = sorted(grid_configs, key=lambda c: (-c["avg_f1"], c["avg_iters"]))[0]
    display(HTML(build_search_report(grid_configs, _best)))

## 6. Ground-Truth Evaluation

Runs the pipeline against three expert-curated scenarios and compares output to known-correct argument structures. Three metrics:

- **Recall** = `|matched| / |GT|` — fraction of ground truth elements found in pipeline output. Primary threshold: ≥ 60%.
- **Level Accuracy** — for matched elements with a strength/confidence axis, fraction with the correct level assigned. Threshold: ≥ 70%.
- **F1** — harmonic mean of recall and precision. Informational — the pipeline legitimately generates more nodes than the minimum GT specifies.

**Matching logic** (`_match_gt_element`): a pipeline element matches a GT element when it shares the same node type (3 points) plus at least 1 content word in common above stopwords (total score ≥ 4).

In [4]:
import re
import json
import csv
from agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP
from argument_templates import TEMPLATE_VERSION

# ── Ground truth — loaded from ground_truth.csv ───────────────────────────
GT_CSV = Path("ground_truth.csv")
GROUND_TRUTH = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        GROUND_TRUTH.append({
            "name":               row["Scenario"],
            "verified_analysis":  json.loads(row["verified_analysis"]),
            "expected_elements":  json.loads(row["expected_elements"]),
            "expected_key_finding": row["expected_key_finding"],
        })

# ── Element extraction and matching ──────────────────────────────────────────
_NODE_PATTERNS = {
    "conclusion":  re.compile(r"\b(C-\d+)\b", re.IGNORECASE),
    "inference":   re.compile(r"\b(I-\d+)\b", re.IGNORECASE),
    "assumption":  re.compile(r"\b(A-\d+)\b", re.IGNORECASE),
    "alternative": re.compile(r"\b(ALT-\d+)\b", re.IGNORECASE),
    "evidence":    re.compile(r"\b(E-\d+)\b", re.IGNORECASE),
}

_LEVEL_KEYWORDS = {
    "HIGH":     ["high confidence", "high"],
    "MEDIUM":   ["medium confidence", "medium"],
    "LOW":      ["low confidence", "low"],
    "STRONG":   ["strong", "[strong]"],
    "MODERATE": ["moderate", "[moderate]"],
    "WEAK":     ["weak", "[weak]"],
}

_LEVEL_PATTERNS = {
    level: [re.compile(r"\b" + re.escape(kw) + r"\b", re.IGNORECASE) for kw in keywords]
    for level, keywords in _LEVEL_KEYWORDS.items()
}

def _extract_elements_from_argument_map(argument_map: str) -> list[dict]:
    """Extract argument elements from the argument map text."""
    elements = []
    seen: set[str] = set()
    lines = argument_map.split("\n")
    for line_idx, line in enumerate(lines):
        for node_type, pattern in _NODE_PATTERNS.items():
            match = pattern.search(line)
            if match:
                node_id = match.group(1).upper()
                if node_id in seen:
                    break
                seen.add(node_id)
                window_end = min(line_idx + 4, len(lines))
                window = " ".join(lines[line_idx:window_end])
                level = "UNKNOWN"
                for level_name, level_pats in _LEVEL_PATTERNS.items():
                    if any(p.search(window) for p in level_pats):
                        level = level_name
                        break
                elements.append({
                    "node_id": node_id,
                    "type": node_type,
                    "level": level,
                    "line": line.strip()[:300],
                })
                break
    return elements


def _match_gt_element(gt_key, gt_elem, pipeline_elements):
    """Find best-matching pipeline element for a GT element."""
    gt_type = gt_elem.get("type", "unknown")
    gt_desc = gt_elem.get("description", "").lower()
    _STOPWORDS = {"the", "a", "an", "is", "was", "were", "are", "with", "from",
                  "for", "and", "or", "not", "in", "on", "to", "of", "by", "at"}
    gt_words = {w.strip(".,;:()[]") for w in gt_desc.split()} - _STOPWORDS
    best_match = None
    best_score = 0
    for pe in pipeline_elements:
        score = 3 if pe["type"] == gt_type else 0
        pe_text = pe.get("line", "").lower()
        pe_words = {w.strip(".,;:()[]") for w in pe_text.split()} - _STOPWORDS
        score += len(gt_words & pe_words)
        if score > best_score:
            best_score = score
            best_match = pe
    return best_match if best_score >= 4 else None


def compare_to_ground_truth(pipeline_elements, gt_elements):
    """Compare pipeline output to expert-curated ground truth."""
    gt_keys = list(gt_elements.keys())
    overlap, missed, matched_indices = [], [], set()
    level_correct = level_total = 0
    for gt_key in gt_keys:
        gt_elem = gt_elements[gt_key]
        match = _match_gt_element(gt_key, gt_elem, pipeline_elements)
        if match:
            overlap.append(gt_key)
            for idx, pe in enumerate(pipeline_elements):
                if pe is match:
                    matched_indices.add(idx)
                    break
            expected_level = gt_elem.get("strength", gt_elem.get("confidence", None))
            if expected_level is not None:
                actual_level = match.get("level", "UNKNOWN")
                level_total += 1
                if actual_level.upper() == expected_level.upper():
                    level_correct += 1
        else:
            missed.append(gt_key)
    extra = [pe.get("node_id", "?") for idx, pe in enumerate(pipeline_elements)
             if idx not in matched_indices]
    recall = len(overlap) / len(gt_keys) if gt_keys else 1.0
    precision = len(overlap) / len(pipeline_elements) if pipeline_elements else 1.0
    f1 = (2 * recall * precision / (recall + precision)) if (recall + precision) > 0 else 0.0
    level_accuracy = (level_correct / level_total * 100) if level_total > 0 else 100.0
    return {
        "overlap": overlap, "missed": missed, "extra": extra,
        "recall": round(recall, 3), "precision": round(precision, 3),
        "f1": round(f1, 3), "level_accuracy": round(level_accuracy, 1),
        "gt_count": len(gt_keys), "pipeline_count": len(pipeline_elements),
    }

print(f"Pipeline model         : {DEFAULT_MODEL}")
print(f"Pipeline temperature   : {DEFAULT_TEMP}")
print(f"Template version       : {TEMPLATE_VERSION}")
print(f"Ground truth scenarios : {len(GROUND_TRUTH)}")
for i, gt in enumerate(GROUND_TRUTH, 1):
    n_expected = len(gt["expected_elements"])
    print(f"  {i}. {gt['name']}  ({n_expected} expected elements)")

Pipeline model         : gemini-2.5-flash
Pipeline temperature   : 0.2
Template version       : 1.0.0
Ground truth scenarios : 3
  1. AiTM Session Hijacking — Full Argument  (5 expected elements)
  2. Ransomware via Stolen VPN Credentials  (5 expected elements)
  3. BEC with OAuth Consent Abuse  (5 expected elements)


In [5]:
from google.adk.sessions import InMemorySessionService

# Use InMemory for notebook eval — no SQLite WAL contention, sessions don't
# need to persist across runs.
_nb_session_svc = InMemorySessionService()

async def run_ground_truth_eval():
    results = []
    for i, gt in enumerate(GROUND_TRUTH, 1):
        print(f"\nScenario {i}/{len(GROUND_TRUTH)}: {gt['name']}")
        _, iterations = await run_pipeline(
            verified_analysis=gt["verified_analysis"],
            max_iterations=3,
            session_service=_nb_session_svc,
        )
        final_verdict = iterations[-1]["verdict"]
        argument_map_text = iterations[-1].get("argument_map", "")
        elements = _extract_elements_from_argument_map(argument_map_text)
        comparison = compare_to_ground_truth(elements, gt["expected_elements"])
        verdict_label = final_verdict.get("verdict", "UNKNOWN")
        n_iters = len(iterations)

        results.append({
            "name": gt["name"],
            "comparison": comparison,
            "n_iters": n_iters,
            "verdict": verdict_label,
        })

        print(f"  Verdict     : {verdict_label} in {n_iters} iteration(s)")
        print(f"  Elements    : pipeline={comparison['pipeline_count']}  gt={comparison['gt_count']}")
        print(f"  Recall      : {comparison['recall']:.1%}")
        print(f"  Level acc   : {comparison['level_accuracy']:.0f}%")
        print(f"  F1          : {comparison['f1']:.3f}")
        if comparison["missed"]:
            for key in comparison["missed"]:
                desc = gt["expected_elements"][key]["description"]
                print(f"  MISSED      : {key} — {desc}")
    return results

gt_results = await run_ground_truth_eval()


Scenario 1/3: AiTM Session Hijacking — Full Argument
  [TLP CHECK] TLP: No 'Product TLP:' declaration found. Source TLPs found: AMBER
  Iteration 1/3: PASS | TLP issues: 1 — The intelligence product is structurally sound, adheres to all logic rules, and includes all required sections.
  Verdict     : PASS in 1 iteration(s)
  Elements    : pipeline=16  gt=5
  Recall      : 100.0%
  Level acc   : 33%
  F1          : 0.476

Scenario 2/3: Ransomware via Stolen VPN Credentials
  Iteration 1/3: PASS — All required sections are present, and all argument nodes and their connections adhere to the specified logic rules.
  Verdict     : PASS in 1 iteration(s)
  Elements    : pipeline=10  gt=5
  Recall      : 20.0%
  Level acc   : 100%
  F1          : 0.133
  MISSED      : inference_moderate_cred_theft — Credentials stolen from breach (2 evidence nodes)
  MISSED      : inference_strong_compromise_chain — Full compromise chain (3+ evidence, 3+ categories)
  MISSED      : assumption_breach_current 

In [9]:
import csv as _csv

RECALL_THRESHOLD = 0.60
LEVEL_THRESHOLD  = 70.0

avg_recall = sum(r["comparison"]["recall"] for r in gt_results) / len(gt_results)
avg_level  = sum(r["comparison"]["level_accuracy"] for r in gt_results) / len(gt_results)
avg_f1     = sum(r["comparison"]["f1"] for r in gt_results) / len(gt_results)

# ── Styled table ─────────────────────────────────────────────────────────────────────────

def _style_gt(rows: list[dict]):
    def row_color(row):
        r_ok = row["Recall"]    >= RECALL_THRESHOLD
        l_ok = row["Lvl Acc %"] >= LEVEL_THRESHOLD
        if r_ok and l_ok:
            return ["background-color: #dcfce7"] * len(row)
        # ground-truth FAIL: red fill PLUS bold text and a solid left border so
        # the row stays distinguishable in black-and-white print where the
        # red/green fill is invisible. Each row is also marked ✓/✗ in GT Result.
        return ["background-color: #fee2e2; font-weight: 700; "
                "border-left: 3px solid #1e293b"] * len(row)

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
        }.get(val, "")

    df = pd.DataFrame([{
        "Scenario":   r["name"],
        "GT":         r["comparison"]["gt_count"],
        "Pipeline":   r["comparison"]["pipeline_count"],
        "Overlap":    len(r["comparison"]["overlap"]),
        "Missed":     len(r["comparison"]["missed"]),
        "Extra":      len(r["comparison"]["extra"]),
        "Recall":     r["comparison"]["recall"],
        "Precision":  r["comparison"]["precision"],
        "Lvl Acc %":  r["comparison"]["level_accuracy"],
        "F1":         r["comparison"]["f1"],
        "Verdict":    r["verdict"],
        "GT Result":  ("✓ OK" if (r["comparison"]["recall"] >= RECALL_THRESHOLD
                                   and r["comparison"]["level_accuracy"] >= LEVEL_THRESHOLD)
                       else "✗ FAIL"),
        "Iters":      r["n_iters"],
    } for r in rows])

    r_ok = avg_recall >= RECALL_THRESHOLD
    l_ok = avg_level  >= LEVEL_THRESHOLD
    status = "PASS" if (r_ok and l_ok) else "FAIL"

    _styler = df.style.apply(row_color, axis=1)
    _map = _styler.map if hasattr(_styler, "map") else _styler.applymap
    return (
        _map(verdict_color, subset=["Verdict"])
        .format({
            "Recall":    "{:.1%}",
            "Precision": "{:.1%}",
            "Lvl Acc %": "{:.0f}%",
            "F1":        "{:.3f}",
        })
        .set_caption(
            f"Ground-Truth Evaluation — avg Recall={avg_recall:.1%} "
            f"(threshold {RECALL_THRESHOLD:.0%}) | "
            f"avg Level Accuracy={avg_level:.0f}% "
            f"(threshold {LEVEL_THRESHOLD:.0f}%) — {status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )

display(_style_gt(gt_results))

# ── Pass/fail thresholds ─────────────────────────────────────────────────────────────────────
print()
print(f"avg Recall           : {avg_recall:.1%}  (threshold {RECALL_THRESHOLD:.0%})"
      f"  {'PASS' if avg_recall >= RECALL_THRESHOLD else 'FAIL'}")
print(f"avg Level Accuracy   : {avg_level:.0f}%  (threshold {LEVEL_THRESHOLD:.0f}%)"
      f"  {'PASS' if avg_level >= LEVEL_THRESHOLD else 'FAIL'}")

# ── Write results back to ground_truth.csv ───────────────────────────────────────────────────────
results_by_name = {r["name"]: r for r in gt_results}
csv_rows = []
with Path("ground_truth.csv").open(newline="", encoding="utf-8") as f:
    reader = _csv.DictReader(f)
    fieldnames = reader.fieldnames
    for row in reader:
        r = results_by_name.get(row["Scenario"])
        if r:
            c = r["comparison"]
            row["Pipeline_Findings"] = c["pipeline_count"]
            row["GT_Findings"]       = c["gt_count"]
            row["Overlap"]           = len(c["overlap"])
            row["Missed"]            = len(c["missed"])
            row["Extra"]             = len(c["extra"])
            row["Recall"]            = c["recall"]
            row["Level_Accuracy"]    = c["level_accuracy"]
            row["F1"]                = c["f1"]
            row["Verdict"]           = r["verdict"]
            row["Iters"]             = r["n_iters"]
        csv_rows.append(row)

with Path("ground_truth.csv").open("w", newline="", encoding="utf-8") as f:
    writer = _csv.DictWriter(f, fieldnames=fieldnames, quoting=_csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"\nWrote results to: ground_truth.csv")

Scenario,GT,Pipeline,Overlap,Missed,Extra,Recall,Precision,Lvl Acc %,F1,Verdict,GT Result,Iters
AiTM Session Hijacking — Full Argument,5,16,5,0,12,100.0%,31.2%,33%,0.476,PASS,✗ FAIL,1
Ransomware via Stolen VPN Credentials,5,10,1,4,9,20.0%,10.0%,100%,0.133,PASS,✗ FAIL,1
BEC with OAuth Consent Abuse,5,15,4,1,11,80.0%,26.7%,67%,0.400,PASS,✗ FAIL,1



avg Recall           : 66.7%  (threshold 60%)  PASS
avg Level Accuracy   : 67%  (threshold 70%)  FAIL

Wrote results to: ground_truth.csv


### Load Existing Ground Truth Results

If you have already run the ground truth eval and want to inspect the saved results without re-running the pipeline, use the cell below.

**Provenance note (2026-07-13).** The recorded ground-truth run for this chapter is the one stored in `ground_truth.csv` and rendered in `Images/Ground Truth Eval.png` (Figure 10.13): average recall 66.7% (PASS at the 60% threshold), average level accuracy 67% (FAIL at the 70% threshold), overall FAIL, all three judge verdicts PASS in 1 iteration. An earlier, superseded run (recall 100/40/60, level accuracy 33/50/50, F1 0.556/0.250/0.300) previously survived in the cached output of the loader cell below; that stale cache has been cleared. Re-running the cell reads the as-run numbers from `ground_truth.csv`. The evaluation cell in section 6 above retains the as-run cached output ("Wrote results to: ground_truth.csv").

In [ ]:
# Load results from ground_truth.csv — reads the result columns written by gt-summary.
import csv as _csv

gt_csv_path = Path("ground_truth.csv")

if gt_csv_path.exists():
    with gt_csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(_csv.DictReader(f))
    has_results = any(row.get("Verdict", "PENDING") != "PENDING" for row in rows)
    if has_results:
        print(f"Template version : {TEMPLATE_VERSION}")
        print()
        for row in rows:
            verdict = row.get("Verdict", "PENDING")
            print(
                f"  {row['Scenario']:<46s}  "
                f"R={float(row['Recall']):.0%}  "
                f"Lvl={float(row['Level_Accuracy']):.0f}%  "
                f"F1={float(row['F1']):.3f}  "
                f"{verdict} ({row['Iters']}it)"
            )
    else:
        print("No results yet — run the ground truth evaluation cells above.")
else:
    print(f"File not found: {gt_csv_path}")

## 7. Judge Calibration

Ten synthetic argument maps with known-correct verdicts test whether the Logic Judge correctly identifies rule violations. The cases cover five logic rules plus structural checks:

| Case | Name | Tests | Expected |
|---|---|---|---|
| 1 | `valid_complete_argument` | All 5 LRs satisfied, all 8 sections present | PASS |
| 2 | `conclusion_no_evidence_chain` | LR-001 + LR-003: single evidence node | FAIL |
| 3 | `inference_no_source_citation` | LR-002: inference with no `E-NNN` citation | FAIL |
| 4 | `strong_with_two_evidence` | LR-003: STRONG label, only 2 evidence nodes | FAIL |
| 5 | `implicit_assumption` | LR-004: assumption embedded in inference prose | PARTIAL |
| 6 | `no_alternative_hypotheses` | LR-005: no `ALT-NNN` nodes | PARTIAL |
| 7 | `circular_reasoning` | LR-002 variant: inference cites another inference | FAIL |
| 8 | `missing_product_sections` | Only 5 of 8 required sections present | PARTIAL |
| 9 | `weak_chain_explicit` | WEAK label correctly applied — judge must accept | PASS |
| 10 | `comprehensive_argument` | Multi-conclusion map, all rules satisfied | PASS |

All 10 cases are scorable. Accuracy threshold: 80% (8/10 must match on both verdict and violation count).

In [8]:
JUDGE_ACCURACY_THRESHOLD = 80.0

print("=" * 70)
print("JUDGE CALIBRATION — Production Stage")
print(f"Test cases : {len(TEST_CASES)}  (all scorable)")
print(f"Threshold  : {JUDGE_ACCURACY_THRESHOLD:.0f}%")
print("=" * 70)

eval_rows   = []
total_pass  = total_fail = total_checks = schema_failures = 0

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n{'─'*60}")
    print(f"Case {i}/{len(TEST_CASES)}: {case['name']}")

    result = await run_judge_on_argument_map(case["argument_map"])

    if not result["valid"]:
        print(f"  SCHEMA FAILURE: {result['error'][:200]}")
        schema_failures += 1
        total_fail      += 1
        total_checks    += 1
        eval_rows.append({
            "name":                case["name"],
            "expected_verdict":    case.get("expected_verdict", "–"),
            "actual_verdict":      "ERROR",
            "expected_violations": case.get("expected_violations", "–"),
            "actual_violations":   "–",
            "verdict_correct":     False,
            "violations_correct":  False,
            "passed":              False,
            "summary":             result.get("error", "")[:80],
        })
        continue

    verdict = result["verdict"]
    scores  = _score_case(case, result)

    n_p = len(scores["passed_checks"])
    n_f = len(scores["failed_checks"])
    total_pass   += n_p
    total_fail   += n_f
    total_checks += n_p + n_f

    actual_verdict     = verdict.get("verdict", "UNKNOWN")
    actual_violations  = len(verdict.get("unverified", []))
    verdict_correct    = actual_verdict == case.get("expected_verdict")
    violations_correct = actual_violations == case.get("expected_violations")

    print(f"  Verdict  : {actual_verdict}  "
          f"(expected={case.get('expected_verdict', '–')})"
          f"  {'OK' if verdict_correct else 'MISMATCH'}")
    print(f"  Unverified: {actual_violations}  "
          f"(expected={case.get('expected_violations', '–')})"
          f"  {'OK' if violations_correct else 'MISMATCH'}")
    for check in scores["passed_checks"]:
        print(f"  + {check}")
    for check in scores["failed_checks"]:
        print(f"  - {check}")

    eval_rows.append({
        "name":                case["name"],
        "expected_verdict":    case.get("expected_verdict", "–"),
        "actual_verdict":      actual_verdict,
        "expected_violations": case.get("expected_violations", "–"),
        "actual_violations":   actual_violations,
        "verdict_correct":     verdict_correct,
        "violations_correct":  violations_correct,
        "passed":              n_f == 0,
        "summary":             verdict.get("summary", "")[:100],
    })

# ── Results summary ──────────────────────────────────────────────────────────────────────────
accuracy = (total_pass / total_checks * 100) if total_checks > 0 else 0
status   = "PASS" if accuracy >= JUDGE_ACCURACY_THRESHOLD else "FAIL"
print(f"\n{'='*70}")
print(f"Checks passed : {total_pass}/{total_checks}  ({total_fail} failed, "
      f"{schema_failures} schema failure(s))")
print(f"Accuracy      : {accuracy:.0f}%  (threshold {JUDGE_ACCURACY_THRESHOLD:.0f}%) — {status}")
print("=" * 70)

JUDGE CALIBRATION — Production Stage
Test cases : 10  (all scorable)
Threshold  : 80%

────────────────────────────────────────────────────────────
Case 1/10: valid_complete_argument
  Verdict  : PASS  (expected=PASS)  OK
  Unverified: 0  (expected=0)  OK
  + Verdict correct: PASS
  + Violation count correct: 0

────────────────────────────────────────────────────────────
Case 2/10: conclusion_no_evidence_chain
  Verdict  : FAIL  (expected=FAIL)  OK
  Unverified: 2  (expected=–)  MISMATCH
  + Verdict correct: FAIL
  + Violation count adequate: 2 >= 1

────────────────────────────────────────────────────────────
Case 3/10: inference_no_source_citation
  Verdict  : FAIL  (expected=FAIL)  OK
  Unverified: 1  (expected=1)  OK
  + Verdict correct: FAIL
  + Violation count correct: 1

────────────────────────────────────────────────────────────
Case 4/10: strong_with_two_evidence
  Verdict  : FAIL  (expected=FAIL)  OK
  Unverified: 1  (expected=1)  OK
  + Verdict correct: FAIL
  + Violation 

In [10]:
# ── Styled table ─────────────────────────────────────────────────────────────────────────

def _style_judge_eval(rows: list[dict]) -> "pd.io.formats.style.Styler":
    def row_color(row):
        # OK rows: light green. FAIL rows: light red PLUS bold text and a solid
        # left border, so the pass/fail split survives black-and-white print
        # where the red/green fill is invisible. Each row is also marked with a
        # ✓ or ✗ in the Result column as a color-independent cue.
        if "FAIL" in row["Result"]:
            return ["background-color: #fee2e2; font-weight: 700; "
                    "border-left: 3px solid #1e293b"] * len(row)
        return ["background-color: #dcfce7"] * len(row)

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(str(val), "color: #64748b")

    df = pd.DataFrame([{
        "Test Case":  r["name"],
        "Expected":   r["expected_verdict"],
        "Actual":     r["actual_verdict"],
        "Exp Unverif":r["expected_violations"],
        "Act Unverif":r["actual_violations"],
        "Result":     "✓ OK" if r["passed"] else "✗ FAIL",
        "Summary":    r["summary"],
    } for r in rows])

    n_pass    = sum(1 for r in rows if r["passed"])
    n_total   = len(rows)
    acc       = n_pass / n_total * 100 if n_total else 0
    threshold = JUDGE_ACCURACY_THRESHOLD
    cap_status = "PASS" if acc >= threshold else "FAIL"

    _styler = df.style.apply(row_color, axis=1)
    _map = _styler.map if hasattr(_styler, "map") else _styler.applymap
    return (
        _map(verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration — {n_pass}/{n_total} cases fully correct "
            f"({acc:.0f}%) — Threshold {threshold:.0f}% — {cap_status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "300px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

display(_style_judge_eval(eval_rows))

Test Case,Expected,Actual,Exp Unverif,Act Unverif,Result,Summary
valid_complete_argument,PASS,PASS,0,0,✓ OK,The intelligence product is structurally sound and adheres to all specified logic rules and required
conclusion_no_evidence_chain,FAIL,FAIL,–,2,✓ OK,The intelligence product fails due to insufficient independent evidence for the conclusion and misca
inference_no_source_citation,FAIL,FAIL,1,1,✓ OK,The intelligence product fails validation due to an inference lacking explicit citation of supportin
strong_with_two_evidence,FAIL,FAIL,1,1,✓ OK,The intelligence product fails validation due to an inference strength miscalibration (LR-003).
implicit_assumption,PARTIAL,PARTIAL,–,2,✓ OK,The product is partially valid; key assumptions are embedded in inference reasoning instead of being
no_alternative_hypotheses,PARTIAL,FAIL,–,1,✗ FAIL,The intelligence product fails due to an undefined supporting inference for a conclusion and a missi
circular_reasoning,FAIL,FAIL,1,5,✗ FAIL,The intelligence product fails due to circular dependencies in inference citations (LR-002) and misc
missing_product_sections,PARTIAL,PARTIAL,0,0,✓ OK,"The intelligence product is structurally sound regarding evidence chains, source citations, inferenc"
weak_chain_explicit,PASS,PASS,0,0,✓ OK,The intelligence product is structurally sound and adheres to all specified logic rules and required
comprehensive_argument,PASS,PARTIAL,0,1,✗ FAIL,The intelligence product is partially valid due to a missing alternative hypothesis for conclusion C


## 8. Discussion

The three evaluations form a layered validation hierarchy, each catching failure modes the others cannot.

**Grid search** establishes whether the self-refining loop converges reliably across diverse analysis scenarios and model configurations. A ceiling effect — F1=100% across all nine configurations — indicates the logic rules fully constrain the output space: the pipeline is rule-limited, not model-limited. When that ceiling is present, the winning configuration is the cheapest one with the lowest average iteration count.

**Ground-truth evaluation** breaks the judge's monopoly on correctness. A lenient judge can emit PASS verdicts on argument maps that miss critical evidence chains. The recall check surfaces this: if the pipeline's confirmed elements don't overlap with what a human analyst would construct, the argument map is structurally wrong regardless of the judge's opinion. Level accuracy adds a second dimension — whether the pipeline's strength ratings are consistent with the evidence weight a human would assign.

**Judge calibration** validates the judge's own reliability before trusting its verdicts in the grid search. If the judge fails calibration — misidentifying rule violations, accepting strength inflation, or missing evidence chain requirements — the F1 scores from the grid search are meaningless. The calibration runs first in any evaluation pipeline for this reason.

**Ceiling interpretation**: When all three evaluations pass with F1=100%, recall above threshold, and judge accuracy above threshold, the production stage is operating within its designed constraints. The self-refining loop is working as intended. The appropriate next step is not further tuning of this stage but integration testing across the full intelligence cycle.

**When to re-run the grid**: If `argument_templates.py` is updated (new template version), new logic rules are added, or the pipeline architecture changes, re-run the grid search to re-evaluate the optimal configuration. The `template_version` field in `best_config.json` will trigger a warning at import if it is stale.

In [ ]:
# Load and display best_config.json after grid search completes.
# Also works if you want to inspect results from a previous run.

best_cfg_path = Path("my_agent/best_config.json")

if best_cfg_path.exists():
    cfg = json.loads(best_cfg_path.read_text())
    print("=" * 60)
    print("BEST CONFIG")
    print("=" * 60)
    print(f"  Model            : {cfg.get('model', '?')}")
    print(f"  Temperature      : {cfg.get('temperature', '?')}")
    print(f"  avg F1           : {cfg.get('avg_f1', '?')}")
    print(f"  avg Recall       : {cfg.get('avg_recall', '?')}")
    print(f"  avg Level acc    : {cfg.get('avg_level_accuracy', '?')}")
    print(f"  Template version : {cfg.get('template_version', '?')}")
    print(f"  Generated at     : {cfg.get('generated_at', '?')}")
    print("=" * 60)

    # Stale-config check
    if cfg.get('template_version') != TEMPLATE_VERSION:
        print(f"[WARN] best_config.json was produced under schema v{cfg.get('template_version')}, "
              f"but current schema is v{TEMPLATE_VERSION}. "
              f"Re-run the grid search cell above to re-evaluate under the updated rules.")
    else:
        print(f"Schema version current (v{TEMPLATE_VERSION}) — no re-run needed.")
else:
    print(f"best_config.json not found at {best_cfg_path}")
    print("Run the grid search cell above to generate it.")

In [ ]:
# Requires the grid-search cells in section 5 to have run first
# (they define SCENARIOS); otherwise this spot-check is skipped.
if "SCENARIOS" not in globals():
    raise SystemExit("Skipped: run the section 5 grid-search cells first (SCENARIOS undefined).")

# Spot-check: run the pipeline once against a single scenario to verify
# the best config is functioning correctly before committing to a full re-run.

SPOT_CHECK_SCENARIO = SCENARIOS[0]  # AiTM with full evidence chain

print(f"Spot-check scenario:")
print(f"  {SPOT_CHECK_SCENARIO[:120]}...")
print(f"  Model: {DEFAULT_MODEL}  Temp: {DEFAULT_TEMP}")
print()

async def spot_check():
    _svc = InMemorySessionService()
    session_id, iterations = await run_pipeline(
        verified_analysis=SPOT_CHECK_SCENARIO,
        max_iterations=MAX_ITERATIONS,
        session_service=_svc,
    )
    final = iterations[-1]
    verdict = final["verdict"]
    elems = _extract_elements_from_argument_map(final.get("argument_map", ""))
    gt = GROUND_TRUTH[0]  # AiTM scenario
    comparison = compare_to_ground_truth(elems, gt["expected_elements"])
    print(f"Verdict    : {verdict.get('verdict', '?')} in {len(iterations)} iteration(s)")
    print(f"Recall     : {comparison['recall']:.1%}")
    print(f"Level acc  : {comparison['level_accuracy']:.0f}%")
    print(f"F1         : {comparison['f1']:.3f}")
    print(f"Overlap    : {len(comparison['overlap'])}/{comparison['gt_count']}")
    if comparison["missed"]:
        print("Missed:")
        for k in comparison["missed"]:
            print(f"  - {k}: {gt['expected_elements'][k]['description']}")
    return iterations

spot_iterations = await spot_check()

## 9. Argument Map Visualization (NetworkX Multilayer DAG)

The argument schema (E → I → C with A and ALT as side-attached nodes) is a
directed acyclic graph by construction. Rendering it through NetworkX makes
the structure verifiable from the same regex patterns the Logic_Judge uses,
so the picture cannot drift from the schema. Five layers map to five node
types: Evidence (bottom), Inference, Conclusion (top), Assumption (right
rail), Alternative Hypothesis (far right). Edge style encodes the relation
type — solid = supports, dashed = assumes, dotted = alternative-to — and
edge width on inference→conclusion edges encodes the inference strength
(WEAK / MODERATE / STRONG).

Several rule violations become visually obvious:
- LR-001 (chain completeness): a C-NNN with fewer than 2 incoming E-paths.
- LR-002 (source citation): an I-NNN with no incoming E-NNN edge.
- LR-003 (strength calibration): a STRONG inference (thick edge) backed by
  fewer than 3 evidence nodes.
- LR-005 (alternatives): a C-NNN with no incoming dotted ALT edge.


In [44]:
import re
import textwrap
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Node-ID patterns mirror agent.py.
_RE_E   = re.compile(r"\bE-\d+\b",   re.IGNORECASE)
_RE_I   = re.compile(r"\bI-\d+\b",   re.IGNORECASE)
_RE_C   = re.compile(r"\bC-\d+\b",   re.IGNORECASE)
_RE_A   = re.compile(r"\bA-\d+\b",   re.IGNORECASE)
_RE_ALT = re.compile(r"\bALT-\d+\b", re.IGNORECASE)

_DEF_LINE = re.compile(
    r"^[ \t]*[-*•][ \t]+\(?(ALT-\d+|[EIAC]-\d+)\)?\b",
    re.MULTILINE | re.IGNORECASE,
)

_STRENGTH = {"STRONG": 3.0, "MODERATE": 2.0, "WEAK": 1.0}

NODE_STYLE = {
    "E":   dict(face="#E5EBE2", edge="#5C7A5E", layer=0, label="Evidence"),
    "I":   dict(face="#E2E8EE", edge="#4A6178", layer=1, label="Inference"),
    "C":   dict(face="#EEE0E0", edge="#8A4F4F", layer=2, label="Conclusion"),
    "A":   dict(face="#EEE9D9", edge="#9A8047", layer=3, label="Assumption"),
    "ALT": dict(face="#E4E5E7", edge="#5E6770", layer=4, label="Alternative"),
}

EDGE_STYLE = {
    "supports":    dict(color="#3D4852", style="-",  label="supports"),
    "assumes":     dict(color="#9A8047", style="--", label="assumes"),
    "alternative": dict(color="#5E6770", style=":",  label="alternative"),
}

def _kind(nid: str) -> str:
    nid = nid.upper()
    return "ALT" if nid.startswith("ALT-") else nid.split("-", 1)[0]


def parse_argument_map(text: str) -> nx.DiGraph:
    """Parse an argument-map markdown into a directed graph.

    Edges always point supporter -> supported:
        E-NNN -> I-NNN  (evidence supports inference)
        I-NNN -> C-NNN  (inference supports conclusion)
        A-NNN -> I/C    (assumption underlies inference or conclusion)
        ALT-NNN -> C    (alternative challenges conclusion)

    Each node also carries a `description` attribute — the cleaned text
    of its definition window — used for hover tooltips.
    """
    G = nx.DiGraph()
    for rx in (_RE_ALT, _RE_E, _RE_I, _RE_C, _RE_A):
        for m in rx.finditer(text):
            nid = m.group(0).upper()
            if nid not in G:
                G.add_node(nid, kind=_kind(nid),
                           layer=NODE_STYLE[_kind(nid)]["layer"],
                           description="")

    defs = list(_DEF_LINE.finditer(text))
    for i, m in enumerate(defs):
        owner = m.group(1).upper()
        win_start = m.start()
        win_end   = defs[i + 1].start() if i + 1 < len(defs) else len(text)
        full_window = text[win_start:win_end]
        body_window = text[m.end():win_end]
        owner_kind = _kind(owner)

        # Description for hover: cleaned full bullet block, length-capped.
        desc = re.sub(r"\s+", " ", full_window).strip()
        if len(desc) > 360:
            desc = desc[:357] + "…"
        G.nodes[owner]["description"] = desc

        # Strength label.
        line_end = text.find("\n", m.start())
        head = text[m.start(): line_end if line_end != -1 else len(text)]
        for lbl, w in _STRENGTH.items():
            if lbl in head.upper():
                G.nodes[owner]["strength"] = lbl
                G.nodes[owner]["weight"]   = w
                break

        e_refs   = {m.group(0).upper() for m in _RE_E.finditer(body_window)}
        i_refs   = {m.group(0).upper() for m in _RE_I.finditer(body_window)}
        a_refs   = {m.group(0).upper() for m in _RE_A.finditer(body_window)}
        alt_refs = {m.group(0).upper() for m in _RE_ALT.finditer(body_window)}

        if owner_kind == "I":
            for r in e_refs:
                G.add_edge(r, owner, kind="supports")
            for r in a_refs - {owner}:
                G.add_edge(r, owner, kind="assumes")
        elif owner_kind == "C":
            for r in i_refs - {owner}:
                G.add_edge(r, owner, kind="supports",
                           weight=G.nodes[r].get("weight", 1.5))
            for r in a_refs - {owner}:
                G.add_edge(r, owner, kind="assumes")
            for r in alt_refs - {owner}:
                G.add_edge(r, owner, kind="alternative")
        elif owner_kind == "ALT":
            for r in {m.group(0).upper() for m in _RE_C.finditer(body_window)}:
                G.add_edge(owner, r, kind="alternative")
    return G


def _layout(G: nx.DiGraph) -> dict:
    """Prefer Graphviz `dot`; fall back to multipartite."""
    try:
        from networkx.drawing.nx_pydot import graphviz_layout
        return graphviz_layout(
            G, prog="dot",
            args='-Grankdir=BT -Gnodesep=0.45 -Granksep=0.9 -Gsplines=spline',
        )
    except Exception:
        pos = nx.multipartite_layout(G, subset_key="layer", align="horizontal")
        return {n: (x, -y) for n, (x, y) in pos.items()}


def _draw_node(ax, x, y, label, style, w=0.9, h=0.42):
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.4, edgecolor=style["edge"],
        facecolor=style["face"], zorder=2,
    ))
    ax.text(x, y, label, ha="center", va="center",
            fontsize=9, fontweight="bold", color=style["edge"], zorder=3)


def visualize_argument_map(text: str, title: str | None = None,
                           figsize=(13, 8), dpi=140, ax=None,
                           save_path: str | None = None):
    """Static matplotlib renderer (book-quality PNG output)."""
    G = parse_argument_map(text)
    if not G:
        raise ValueError("No argument-map nodes found in input text.")

    pos = _layout(G)
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    else:
        fig = ax.figure

    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    x_span = max(xs) - min(xs) or 1.0
    y_span = max(ys) - min(ys) or 1.0
    pos = {n: ((x - min(xs)) / x_span * 10,
               (y - min(ys)) / y_span * 6) for n, (x, y) in pos.items()}

    for u, v, d in G.edges(data=True):
        es = EDGE_STYLE[d["kind"]]
        lw = 1.2 + 0.6 * d.get("weight", 1.0)
        ax.add_patch(FancyArrowPatch(
            posA=pos[u], posB=pos[v],
            arrowstyle="-|>", mutation_scale=16,
            connectionstyle="arc3,rad=0.12",
            color=es["color"], linestyle=es["style"],
            linewidth=lw, zorder=1, shrinkA=20, shrinkB=20,
        ))
    for n, d in G.nodes(data=True):
        x, y = pos[n]
        _draw_node(ax, x, y, n, NODE_STYLE[d["kind"]])

    pad_x, pad_y = 0.8, 0.7
    ax.set_xlim(min(p[0] for p in pos.values()) - pad_x,
                max(p[0] for p in pos.values()) + pad_x)
    ax.set_ylim(min(p[1] for p in pos.values()) - pad_y,
                max(p[1] for p in pos.values()) + pad_y)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_axis_off()

    legend = [Patch(facecolor=s["face"], edgecolor=s["edge"], label=s["label"])
              for s in NODE_STYLE.values()] + [
        Line2D([0],[0], color=es["color"], lw=2, linestyle=es["style"],
               label=es["label"]) for es in EDGE_STYLE.values()
    ]
    ax.legend(handles=legend, loc="lower center", ncol=4, fontsize=8.5,
              frameon=False, bbox_to_anchor=(0.5, -0.02))
    ax.set_title(title or "Argument Map — Multilayer DAG",
                 fontsize=12, fontweight="bold", pad=12)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    return G, ax



def visualize_argument_map_interactive(text, title=None, height=700,
                                       save_html=None, save_png=None):
    """Interactive Plotly renderer with per-node hover tooltips.

    Hover any node to see its full definition. Renders inline in Jupyter.
    Uses Graphviz `dot` for layout (via networkx.nx_pydot), with the
    multipartite layout as fallback if pydot/dot is unavailable.
    """
    import plotly.graph_objects as go
    import textwrap as _tw

    G = parse_argument_map(text)
    if not G:
        raise ValueError("No argument-map nodes found in input text.")

    pos = _layout(G)
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    x_span = max(xs) - min(xs) or 1.0
    y_span = max(ys) - min(ys) or 1.0
    pos = {n: ((x - min(xs)) / x_span * 10,
               (y - min(ys)) / y_span * 6) for n, (x, y) in pos.items()}

    # Edge traces (one per relation kind) + arrowhead annotations.
    edge_traces = []
    annotations = []
    for kind, es in EDGE_STYLE.items():
        eX, eY = [], []
        for u, v, d in G.edges(data=True):
            if d["kind"] != kind:
                continue
            x0, y0 = pos[u]; x1, y1 = pos[v]
            eX += [x0, x1, None]
            eY += [y0, y1, None]
            annotations.append(dict(
                ax=x0, ay=y0, x=x1, y=y1,
                xref="x", yref="y", axref="x", ayref="y",
                showarrow=True, arrowhead=3, arrowsize=1.6, arrowwidth=1.4,
                arrowcolor=es["color"], standoff=20, startstandoff=20,
                opacity=0.9,
            ))
        if not eX:
            continue
        dash = {"-": "solid", "--": "dash", ":": "dot"}[es["style"]]
        edge_traces.append(go.Scatter(
            x=eX, y=eY, mode="lines",
            line=dict(color=es["color"], width=2, dash=dash),
            name=es["label"], hoverinfo="skip",
        ))

    # Node traces (one per kind for color/legend).
    node_traces = []
    for kind, style in NODE_STYLE.items():
        nX, nY, labels, hovers = [], [], [], []
        for n, d in G.nodes(data=True):
            if d["kind"] != kind:
                continue
            x, y = pos[n]
            nX.append(x); nY.append(y); labels.append(n)
            strength = d.get("strength")
            strength_line = (f"<br><b>Strength/Confidence:</b> {strength}"
                             if strength else "")
            desc = d.get("description", "(no description)")
            wrapped = "<br>".join(
                _tw.wrap(desc, width=45, break_long_words=True,
                         break_on_hyphens=True)
            ) or "(no description)"
            hovers.append(
                f"<b>{n}</b> ({style['label']}){strength_line}"
                f"<br><br>{wrapped}"
            )
        if not nX:
            continue
        node_traces.append(go.Scatter(
            x=nX, y=nY, mode="markers+text",
            text=labels, textposition="middle center",
            textfont=dict(size=11, color=style["edge"], family="Arial Black"),
            marker=dict(size=42, color=style["face"],
                        line=dict(width=2, color=style["edge"]), symbol="square"),
            name=style["label"],
            hovertext=hovers, hoverinfo="text",
            hoverlabel=dict(bgcolor="white", font=dict(size=11, family="Arial"),
                            bordercolor=style["edge"], align="left",
                            namelength=-1),
        ))

    fig = go.Figure(data=edge_traces + node_traces)
    fig.update_layout(
        title=dict(text=title or "Argument Map — Interactive",
                   font=dict(size=16)),
        showlegend=True,
        legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center"),
        annotations=annotations,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False, scaleanchor="x"),
        plot_bgcolor="white", height=height,
        margin=dict(l=20, r=20, t=60, b=80),
        hovermode="closest",
    )
    if save_html:
        _pin_js = """
<script>
(function() {
  function init() {
    var divs = document.getElementsByClassName('plotly-graph-div');
    if (!divs.length) { setTimeout(init, 80); return; }
    var gd = divs[0];
    if (!gd.on) { setTimeout(init, 80); return; }
    var baseAnnotations = (gd.layout.annotations || []).slice();
    var pins = [];
    function rebuild() {
      Plotly.relayout(gd, {annotations: baseAnnotations.concat(pins)});
    }
    gd.on('plotly_click', function(d) {
      var pt = d.points[0];
      if (pt.hovertext === undefined) return;
      var existing = pins.findIndex(function(p) {
        return p.x === pt.x && p.y === pt.y;
      });
      if (existing >= 0) { pins.splice(existing, 1); }
      else {
        pins.push({
          x: pt.x, y: pt.y,
          text: pt.hovertext + '<br><i style=\"color:#888;font-size:10px\">'
                + '(click to unpin)</i>',
          showarrow: true, arrowhead: 2, arrowsize: 1, arrowwidth: 1.2,
          arrowcolor: '#666', ax: 40, ay: -60,
          bgcolor: 'white', bordercolor: '#888', borderwidth: 1,
          borderpad: 8, font: {size: 11, family: 'Arial'},
          align: 'left', xref: 'x', yref: 'y',
          captureevents: true,
        });
      }
      rebuild();
    });
    gd.on('plotly_clickannotation', function(d) {
      var i = d.index;
      if (i >= baseAnnotations.length) {
        pins.splice(i - baseAnnotations.length, 1);
        rebuild();
      }
    });
  }
  init();
})();
</script>
"""
        fig.update_layout(clickmode="event", dragmode=False)
        _html_str = fig.to_html(include_plotlyjs="cdn", full_html=True,
                                config={"displayModeBar": False})
        _html_str = _html_str.replace("</body>", _pin_js + "</body>")
        with open(save_html, "w", encoding="utf-8") as _f:
            _f.write(_html_str)
    if save_png:
        try:
            fig.write_image(save_png, scale=2)
        except Exception as _e:
            print(f"PNG save failed ({_e}); install kaleido: pip install kaleido")
    return G, fig



In [ ]:
# Demo: render the AiTM "valid_complete_argument" sample as an interactive
# argument map (Graphviz dot layout + hover tooltips).
# Substitute any pipeline output — e.g., session.state["verified_product"].

SAMPLE_AITM = """
### 2. Key Findings
- C-001 [HIGH]: AiTM session hijacking via stolen Okta session cookie enabled
  unauthorized GitHub repository access. Supported by I-001, I-002. Key
  assumptions: A-001.
- C-002 [LOW]: The theft was targeted, not opportunistic; the actor sought
  Project Phoenix's pre-release code specifically. Supported by I-003.

### 3. Evidence Summary
- E-001 [Okta System Log, identity_provider]: user.session.start and continued
  activity on the partner developer's existing Okta session from Tor exit IP
  185.220.101.42, a documented anonymizing exit node never previously tied to
  this account. Confidence: HIGH.
- E-002 [SpyCloud, breach_intelligence]: recaptured Okta session cookie for the
  partner user. Confidence: HIGH.
- E-003 [Proofpoint, email_security]: click on Evilginx2 phishing URL targeting
  the partner domain. Confidence: HIGH.
- E-004 [GitHub Audit Log, version_control]: git.clone of the two Project Phoenix
  repos (/phoenix-core, /phoenix-api) from the same Tor exit IP within 45 minutes
  of session start; no comparable clone of the other 40-plus accessible repos.
  Confidence: HIGH.

### 4. Analytical Reasoning
- I-001 [STRONG]: Okta session was established via stolen session cookie from
  AiTM phishing, not legitimate authentication. Supporting evidence: E-001,
  E-002, E-003. Assumes A-001.
- I-002 [MODERATE]: GitHub repository access was unauthorized, using the
  hijacked session. Supporting evidence: E-001, E-004.
- I-003 [WEAK]: The theft was targeted at Project Phoenix specifically, not
  opportunistic access. Supporting evidence: E-004.

### 5. Assumptions and Limitations
- A-001: Okta session from the Tor exit IP was not a legitimate session from the
  partner employee using an anonymization tool.

### 6. Alternative Hypotheses
- ALT-001: Legitimate partner employee using VPN from unusual location.
  Rejected; challenges C-001.
- ALT-002: Opportunistic credential abuse by a non-targeted actor; of 40-plus
  accessible repositories only the two Project Phoenix repos were cloned, a
  selectivity inconsistent with opportunistic exploitation. Rejected; challenges
  C-002.
"""

# Save a static copy to disk for the book figure, without rendering it inline.
import matplotlib.pyplot as plt
_fig, _ax = plt.subplots(figsize=(13, 8), dpi=140)
visualize_argument_map(SAMPLE_AITM,
                       title="AiTM Session Hijacking — Argument Map",
                       save_path="Images/AiTM Argument Map.png",
                       ax=_ax)
plt.close(_fig)  # suppress inline display; PNG is on disk.

# Interactive Plotly view with hover tooltips, also written to disk.
_G, ifig = visualize_argument_map_interactive(
    SAMPLE_AITM,
    title="AiTM Session Hijacking",
    save_html="Images/AiTM Argument Map.html",
    save_png="Images/AiTM Argument Map (interactive).png",
)
print("Saved interactive HTML -> Images/AiTM Argument Map.html")
ifig.show()  # inline view shows hover; for click-to-pin open the saved .html


In [ ]:
# Figure 10.10 render — C7-corrected map (2026-07-13).
# Mirrors the corrected Table 10.3: E-001 recharacterized as the known C2 node
# solely attributed to Actor Group X, new attribution inference I-004, and
# C-002 held at MEDIUM. SAMPLE_AITM above is preserved exactly as run and
# still drives the interactive demo; this cell renders the corrected map
# and overwrites the book-figure master in Images/.

CORRECTED_AITM = """
### 2. Key Findings
- C-001 [HIGH]: AiTM session hijacking via stolen Okta session cookie enabled
  unauthorized GitHub repository access. Supported by I-001, I-002. Key
  assumptions: A-001.
- C-002 [MEDIUM]: The theft was a targeted operation attributed to Actor Group X;
  the actor sought Project Phoenix's pre-release code specifically. Supported by
  I-003, I-004.

### 3. Evidence Summary
- E-001 [Okta System Log, identity_provider]: continued activity on the partner
  developer's existing Okta session from IP 185.220.101.42, with no new
  authentication or MFA event preceding it; Chapter 8's processing pass confirmed
  the IP as a known C2 node solely attributed to Actor Group X, never previously
  tied to this account. Confidence: HIGH.
- E-002 [SpyCloud, breach_intelligence]: recaptured Okta session cookie for the
  partner user. Confidence: HIGH.
- E-003 [Proofpoint, email_security]: click on Evilginx2 phishing URL targeting
  the partner domain. Confidence: HIGH.
- E-004 [GitHub Audit Log, version_control]: git.clone of the two Project Phoenix
  repos (/phoenix-core, /phoenix-api) from the same IP within 45 minutes of
  session start; no comparable clone of the other 40-plus accessible repos.
  Confidence: HIGH.

### 4. Analytical Reasoning
- I-001 [STRONG]: Okta session was established via stolen session cookie from
  AiTM phishing, not legitimate authentication. Supporting evidence: E-001,
  E-002, E-003. Assumes A-001.
- I-002 [MODERATE]: GitHub repository access was unauthorized, using the
  hijacked session. Supporting evidence: E-001, E-004.
- I-003 [WEAK]: The theft was targeted at Project Phoenix specifically, not
  opportunistic access. Supporting evidence: E-004.
- I-004 [WEAK]: Attribution: the session originated from a C2 node solely
  attributed to Actor Group X, a confirmed infrastructure match under Chapter 8's
  CR-005 standard. Supporting evidence: E-001.

### 5. Assumptions and Limitations
- A-001: The session cookie was still active when the GitHub access occurred;
  testable via GitHub OAuth token-creation logs.

### 6. Alternative Hypotheses
- ALT-001: Legitimate partner employee using VPN from unusual location; partner
  IT confirmed no corporate VPN egress from 185.220.101.42, and the IP is
  catalogued as Actor Group X command-and-control infrastructure. Rejected;
  challenges C-001.
- ALT-002: Opportunistic credential abuse by a non-targeted actor; of 40-plus
  accessible repositories only the two Project Phoenix repos were cloned, a
  selectivity inconsistent with opportunistic exploitation. Rejected; challenges
  C-002.
"""

_fig, _ax = plt.subplots(figsize=(13, 8), dpi=140)
visualize_argument_map(CORRECTED_AITM,
                       title="AiTM Session Hijacking — Argument Map",
                       save_path="Images/AiTM Argument Map.png",
                       ax=_ax)
plt.close(_fig)
print("Saved corrected Figure 10.10 master -> Images/AiTM Argument Map.png")
